In [1]:
# ==========================================
# RL_DDM SYNTHETIC DATA GENERATOR
# ==========================================

import numpy as np
import pandas as pd
import os
import sys
import glob
from tqdm import tqdm

# ------------------------------------------
# PATH SETUP
# ------------------------------------------

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

from utils.twostep_support import *
from models import *


In [2]:
import os
print(os.getcwd())

c:\Users\user\OneDrive\Desktop\data\McGill\Neur_503_computational-neuroscience\final_paper\aif-ddm-main


In [3]:
template_game = pd.read_csv(
    "two_step_task_datasets/synthetic_dataset/00001_game.csv"
)

In [4]:
# -------------------------
# Simulation configuration
# -------------------------

model = "RL_ddm"
learning = "RL"
drmtype = "linear"

n_hc = 40
n_ps = 40
n_total = n_hc + n_ps
T = len(template_game)  # magic_carpet_2020 trial count

In [5]:
task_dict = {
    "type": "drift",
    "x": False,
    "r": True,
    "delta": 0.025,
    "bounds": [0.25, 0.75],
    "T": T
}

In [6]:
# -------------------------
# Healthy Controls (RL_DDM)
# -------------------------
hc_means = {
    "Synthetic_lr1": 0.4,          # moderate first-stage learning
    "Synthetic_lr2": 0.4,          # moderate second-stage learning
    "Synthetic_lam": 2.5,          # moderate reward sensitivity
    "Synthetic_w": 0.7,            # strong model-based control
    "Synthetic_p": 0.0,            # no response bias
    "Synthetic_a_bs": 2.0,         # higher boundary separation (cautious)
    "Synthetic_ndt": 0.3,          # normal non-decision time
    "Synthetic_v_stage_0": 4.0,    # efficient value-to-drift mapping
    "Synthetic_v_stage_1": 4.0
}

# -------------------------
# Psychosis Spectrum (RL_DDM)
# -------------------------
patient_means = {
    "Synthetic_lr1": 0.7,          # faster first-stage updating (over-updating)
    "Synthetic_lr2": 0.7,          # faster reward updating
    "Synthetic_lam": 3.5,          # stronger reward impact
    "Synthetic_w": 0.3,            # reduced model-based control
    "Synthetic_p": 0.0,
    "Synthetic_a_bs": 1.2,         # reduced boundary (impulsivity)
    "Synthetic_ndt": 0.3,
    "Synthetic_v_stage_0": 2.5,    # noisier value-to-drift mapping
    "Synthetic_v_stage_1": 2.5
}



In [7]:
def sample_group(means, n, label):
    rows = []
    
    # Parameter-specific standard deviations
    sd_dict = {
        "Synthetic_lr1": 0.08,
        "Synthetic_lr2": 0.08,
        "Synthetic_lam": 0.4,
        "Synthetic_w": 0.08,
        "Synthetic_p": 0.05,
        "Synthetic_a_bs": 0.25,
        "Synthetic_ndt": 0.05,
        "Synthetic_v_stage_0": 0.5,
        "Synthetic_v_stage_1": 0.5
    }
    
    # Parameter bounds
    bounds = {
        "Synthetic_lr1": (0, 1),
        "Synthetic_lr2": (0, 1),
        "Synthetic_lam": (0, 10),
        "Synthetic_w": (0, 1),
        "Synthetic_p": (-1, 1),
        "Synthetic_a_bs": (0.3, 4),
        "Synthetic_ndt": (0, 1),
        "Synthetic_v_stage_0": (0, 10),
        "Synthetic_v_stage_1": (0, 10)
    }

    for i in range(n):
        row = {}

        for param, mean in means.items():
            
            sd = sd_dict[param]
            value = np.random.normal(mean, sd)

            # Enforce bounds
            lower, upper = bounds[param]
            value = np.clip(value, lower, upper)

            row[param] = value

        row["Group"] = label
        rows.append(row)

    return pd.DataFrame(rows)


df_hc = sample_group(hc_means, n_hc, "HC")
df_ps = sample_group(patient_means, n_ps, "PS")

df_param_sets = pd.concat([df_hc, df_ps], ignore_index=True)
df_param_sets["ParticipantID"] = np.arange(1, len(df_param_sets)+1)

df_param_sets.head()



,Synthetic_lr1,Synthetic_lr2,Synthetic_lam,Synthetic_w,Synthetic_p,Synthetic_a_bs,Synthetic_ndt,Synthetic_v_stage_0,Synthetic_v_stage_1,Group,ParticipantID
0,0.356209,0.351614,2.174757,0.705773,0.060596,2.130424,0.351716,3.281902,3.183687,HC,1
1,0.347052,0.324264,3.032344,0.662132,-0.070136,1.974339,0.337485,4.044639,2.738802,HC,2
2,0.290363,0.512979,2.521422,0.857803,-0.020332,1.493523,0.325919,4.097951,4.029851,HC,3
3,0.414461,0.375774,2.509392,0.497861,-0.074267,2.105985,0.252854,4.081719,3.572039,HC,4
4,0.437902,0.210825,3.369180,0.717278,-0.041026,2.273898,0.239745,4.636090,3.414816,HC,5


In [8]:
df_param_sets.to_csv(
    "parameter_recovery/grouped_true_parameters_RL-DDM.csv",
    index=False
)

In [9]:
simulation_results = []

for i in tqdm(range(len(df_param_sets))):
    
    row = df_param_sets.iloc[i]
    
    model_dict = {
        "act": model,
        "learn": learning,
        "drmtype": drmtype,
        "learn_transitions": True,
        
        "lr1": row["Synthetic_lr1"],
        "lr2": row["Synthetic_lr2"],
        "lam": row["Synthetic_lam"],
        "w": row["Synthetic_w"],
        "p": row["Synthetic_p"],
        "a_bs": row["Synthetic_a_bs"],
        "ndt": row["Synthetic_ndt"],
        "v_stage_0": row["Synthetic_v_stage_0"],
        "v_stage_1": row["Synthetic_v_stage_1"]
    }
    
    agent = learn_and_act(task=task_dict, model=model_dict)
    
    actions, observations, rts, pi, p_trans, p_r, Gs = agent.perform_task()
    
    df_participant = pd.DataFrame({
        "Synthetic_participant_ID": row["ParticipantID"],
        "Group": row["Group"],
        "trial": np.arange(T),
        "choice1": actions[:,0],
        "choice2": actions[:,1],
        "rt1": rts[:,0],
        "rt2": rts[:,1],
        "final_state": observations[:,0],
        "reward": observations[:,1]
    })
    
    simulation_results.append(df_participant)

df_simulated = pd.concat(simulation_results, ignore_index=True)

df_simulated.head()



100%|██████████| 80/80 [01:04<00:00,  1.24it/s]


,Synthetic_participant_ID,Group,trial,choice1,choice2,rt1,rt2,final_state,reward
0,1,HC,0,1,0,3.684475,3.613852,0,0
1,1,HC,1,0,0,5.373426,2.503583,0,1
2,1,HC,2,0,1,2.371179,7.336786,1,0
3,1,HC,3,0,0,3.409326,1.876351,0,0
4,1,HC,4,0,1,3.521604,2.395413,1,1


In [10]:
df_simulated_hc = df_simulated[df_simulated['Group'] == 'HC']
df_simulated_patient = df_simulated[df_simulated['Group'] == 'PS']

df_simulated_patient

,Synthetic_participant_ID,Group,trial,choice1,choice2,rt1,rt2,final_state,reward
8040,41,PS,0,0,1,1.621535,8.597613,0,0
8041,41,PS,1,0,0,6.423282,2.216531,0,1
8042,41,PS,2,0,0,1.728577,6.757157,1,1
8043,41,PS,3,0,0,0.929673,2.005223,0,0
8044,41,PS,4,1,0,9.612271,0.710037,1,0
...,...,...,...,...,...,...,...,...,...
16075,80,PS,196,0,1,0.340287,0.502267,0,1
16076,80,PS,197,0,0,0.346119,0.464466,1,0
16077,80,PS,198,0,1,0.321511,0.544516,0,1
16078,80,PS,199,0,0,0.345260,1.478986,1,1


In [11]:
import os
import shutil
import pandas as pd

project_root = r"C:\Users\user\OneDrive\Desktop\data\McGill\Neur_503_computational-neuroscience\final_paper\aif-ddm-main"

template_game_path = os.path.join(
    project_root,
    "two_step_task_datasets",
    "magic_carpet_2020_dataset",
    "choices",
    "03187_game.csv"
)

template_config_path = os.path.join(
    project_root,
    "two_step_task_datasets",
    "magic_carpet_2020_dataset",
    "choices",
    "03187_config.txt"
)

template_tutorial_path = os.path.join(
    project_root,
    "two_step_task_datasets",
    "magic_carpet_2020_dataset",
    "choices",
    "03187_tutorial.csv"
)

template_game = pd.read_csv(template_game_path)
template_tutorial = pd.read_csv(template_tutorial_path)

T = len(template_game)

save_root = os.path.join(
    project_root,
    "two_step_task_datasets",
    "synthetic_dataset_RL-DDM"
)

os.makedirs(save_root, exist_ok=True)

print("Template loaded.")


Template loaded.


In [12]:
for pid in df_simulated["Synthetic_participant_ID"].unique():
    
    df_p = df_simulated[df_simulated["Synthetic_participant_ID"] == pid]
    pid_str = f"{int(pid):05d}"

    # Start from template
    df_game = template_game.copy()

    # Overwrite only behavioural columns
    df_game["choice1"] = df_p["choice1"].values + 1
    df_game["choice2"] = df_p["choice2"].values + 1
    df_game["rt1"] = df_p["rt1"].values
    df_game["rt2"] = df_p["rt2"].values
    df_game["final_state"] = df_p["final_state"].values + 1
    df_game["reward"] = df_p["reward"].values

    # Save game file
    df_game.to_csv(
        os.path.join(save_root, f"{pid_str}_game.csv"),
        index=False
    )

    # Copy config
    shutil.copy(
        template_config_path,
        os.path.join(save_root, f"{pid_str}_config.txt")
    )

    # Copy tutorial
    template_tutorial.to_csv(
        os.path.join(save_root, f"{pid_str}_tutorial.csv"),
        index=False
    )

print("Synthetic dataset written in correct format.")


Synthetic dataset written in correct format.


In [13]:
df_simulated.to_csv("group_simulated_data/synthetic_two_step_RL_ddm_groups.csv", index=False)
df_simulated_hc.to_csv("group_simulated_data/synthetic_two_step_RL_ddm_HC.csv", index=False)
df_simulated_patient.to_csv("group_simulated_data/synthetic_two_step_RL_ddm_patient.csv", index=False)
df_param_sets.to_csv("group_simulated_data/synthetic_true_parameters_RL-DDM.csv", index=False)

print("Simulation complete.")


Simulation complete.
